### 1. First Move

In [18]:
#So here i Imported the pandas lib and the CSV file
import pandas as pd
data = pd.read_csv("dirty_cafe_sales.csv") 

In [10]:
#Nonsense printing the head and tail to see the data to start fixing it 
data.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [11]:
data.tail()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02
9999,TXN_6170729,Sandwich,3,4.0,12.0,Cash,In-store,2023-11-07


In [12]:
#Inspecting data to know the data types and everything
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


### 2. Second move

In [19]:
# Here i didn't like the names of the columns so i changed them to what i see better
data.rename(columns={
    "Transaction ID" : "Transaction_ID",
    "Price Per Unit" : "Price_Per_Unit",
    "Total Spent" :"Total_Spent",
    "Payment Method" : "Payment_Method",
    "Transaction Date" : "Transaction_Date"
}, inplace = True)


In [ ]:
#Here i removed any duplicated row
data = data.drop_duplicates( inplace= True)

### 3. Third move

In [20]:
#So here i pulled the most commonly used item to be filled in my file 
most_frequent = data['Item'].mode()[0]
data['Item'] = data['Item'].fillna(most_frequent)

In [21]:
#Here at first i learned that there is something called to_numeric so i did it here then i chose avg of quantity ordered to be used
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')
data['Quantity'] = data['Quantity'].fillna(data['Quantity'].median())
data['Quantity'] = data['Quantity'].astype('int64')

In [22]:
#here i matched the price with the item column by creating a dict, then them moving into it and match
data['Price_Per_Unit'] = pd.to_numeric(data['Price_Per_Unit'], errors='coerce')

price_menu = {
    'Coffee': 2.0,
    'Tea': 1.5,
    'Sandwich': 4.0,
    'Salad': 5.0,
    'Cake': 3.0,
    'Cookie': 1.0,
    'Smoothie': 4.0,
    'Juice': 3.0
}

data['Price_Per_Unit'] = data['Price_Per_Unit'].fillna(data['Item'].map(price_menu))
data['Price_Per_Unit'] = data['Price_Per_Unit'].fillna(data['Price_Per_Unit'].mean())
data['Price_Per_Unit'] = data['Price_Per_Unit'].astype('float64')

In [23]:
# the total amount of money by multiplying Quantity by Price_Per_Unit
data['Total_Spent'] = data['Quantity'] * data['Price_Per_Unit']

In [24]:
#here looking where cash and digital wallet in payment and replacing them in location and if there is Error replace with the mode
data.loc[(data['Payment_Method'] == 'Cash'), 'Location'] = 'In-store'
data.loc[(data['Payment_Method'] == 'Digital Wallet'), 'Location'] = 'Takeaway'
data['Location'] = data['Location'].replace(['UNKNOWN', 'ERROR'], data['Location'].mode()[0])
data['Location'] = data['Location'].fillna(data['Location'].mode()[0])


In [25]:
#i thought about dropping the empty rows in the Payment_Method but then i chose to reolace them with the mode
common_payment = data['Payment_Method'].mode()[0]
data['Payment_Method'] = data['Payment_Method'].replace(['UNKNOWN', 'ERROR'], common_payment)
data['Payment_Method'] = data['Payment_Method'].fillna(common_payment)

In [26]:
#Changed the data type to date then filling empty values with mode
data['Transaction_Date'] = pd.to_datetime(data['Transaction_Date'], errors='coerce')
data['Transaction_Date'] = data['Transaction_Date'].fillna(data['Transaction_Date'].mode()[0])

In [27]:
# here i learned something new that is called dt toolbox in pandas for dates and month that can see the months from 1-12, also used map
data['Seasons'] = data['Transaction_Date']
season = {
    1: 'Winter', 2: 'Winter', 3: 'Spring',
    4: 'Spring', 5: 'Spring', 6: 'Summer',
    7: 'Summer', 8: 'Summer', 9: 'Autumn',
    10: 'Autumn', 11: 'Autumn', 12: 'Winter'
}
data['Seasons'] = data['Transaction_Date'].dt.month.map(season)


### 4. Fourth move

In [28]:
#printing data now
data

,Transaction_ID,Item,Quantity,Price_Per_Unit,Total_Spent,Payment_Method,Location,Transaction_Date,Seasons
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,Autumn
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,Spring
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,Summer
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,Takeaway,2023-04-27,Spring
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,Takeaway,2023-06-11,Summer
...,...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,Digital Wallet,Takeaway,2023-08-30,Summer
9996,TXN_9659401,Juice,3,3.0,9.0,Digital Wallet,Takeaway,2023-06-02,Summer
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,Takeaway,2023-03-02,Spring
9998,TXN_7695629,Cookie,3,1.0,3.0,Digital Wallet,Takeaway,2023-12-02,Winter


In [29]:
#looking for any null value
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    10000 non-null  str           
 1   Item              10000 non-null  str           
 2   Quantity          10000 non-null  int64         
 3   Price_Per_Unit    10000 non-null  float64       
 4   Total_Spent       10000 non-null  float64       
 5   Payment_Method    10000 non-null  str           
 6   Location          10000 non-null  str           
 7   Transaction_Date  10000 non-null  datetime64[us]
 8   Seasons           10000 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 703.3 KB


In [30]:
#No null values
data.isna().sum()

Transaction_ID      0
Item                0
Quantity            0
Price_Per_Unit      0
Total_Spent         0
Payment_Method      0
Location            0
Transaction_Date    0
Seasons             0
dtype: int64

### 5. Final move

In [ ]:
#Saving data
data.to_csv('Cleaned_Data.csv', index=False)
print("Data is saved")

Data is saved
